# 🔧 Caching, batching & the single-door `LLMClient`

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 11 §11.5–11.7 · type: walkthrough (the chapter's 🔧 Build)*

**One-line promise:** cut the two biggest agent costs — **prompt caching** (repetition within a workload) and **batch APIs** (work that doesn't need an answer now) — then assemble the one internal client every model call flows through.

## 🧠 Why this matters

Two API primitives exploit structure your workload already has. **Caching** attacks repetition *within* a run: an agent re-sends its system prompt, tool definitions, and growing transcript every loop iteration, so the prefill for a stable prefix can be computed once and reused — routinely the single largest cost and latency win for multi-turn agents. **Batching** attacks the other axis: work that tolerates hours of latency (embedding backfills, nightly classification, bulk evals) runs at roughly half price. Paying real-time prices for offline work is donating margin.

Everything in this chapter then converges into the **`LLMClient`** — the single door to model APIs. Model routing, cached prefixes, concurrency control, retries, and usage metrics live in *one* place, so no caller knows (or cares) which provider answered. Part IV's agents are built entirely on top of it.

## Objectives & prereqs

**By the end you can:**
- Set a `cache_control` breakpoint after a stable prefix and read `cache_creation_input_tokens` / `cache_read_input_tokens` from `usage`.
- Model an async batch submit/collect at ~half price and compute the savings.
- Name the per-provider features that belong behind one adapter (citations, hosted tools, cache semantics, reasoning-block round-tripping).
- Stand up the single-door `LLMClient` with model routing, a cached system prefix, a concurrency limiter + circuit breaker, and usage metrics.

**Prereqs:** notebooks **11-01** (response shape) and **11-02** (retries/idempotency); the Ch 10 prompt registry concept (stable-prefix-first layout). No API key needed in mock mode.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv from requirements.txt). asyncio drives the async client.
import os
import asyncio
import random
from dataclasses import dataclass, field

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# MOCK=1 (default): canned usage objects with synthetic cache-hit fields and a
#   canned batch result, so all cost math runs offline & deterministically.
# MOCK=0: exercises real caching on a multi-turn run (cache read/create tokens
#   flagged); the batch demo stays an optional gated cell.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(11)
print(f"MOCK mode: {MOCK}")
if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit("MOCK=0 but ANTHROPIC_API_KEY is unset — add it to .env or stay in mock mode.")

### The mock layer

`usage` now carries the two cache fields the book and the `llm-gateway` blueprint use: `cache_creation_input_tokens` (you paid to *write* the cache on the first call) and `cache_read_input_tokens` (billed at a fraction on every *hit* after). The mock provider simulates a cache keyed on the stable prefix.

In [ ]:
@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    cache_creation_input_tokens: int = 0   # paid once to write the cache
    cache_read_input_tokens: int = 0       # billed ~0.1x on every cache hit


@dataclass
class Block:
    type: str
    text: str | None = None


@dataclass
class Message:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: Usage = field(default_factory=Usage)
    model: str = "claude-sonnet-4-6"


class MockProvider:
    """Simulates prompt caching: the first call with a given stable prefix pays
    cache_creation; later calls with the SAME prefix pay the cheap cache_read."""

    def __init__(self):
        self._cache = {}  # prefix_text -> cached token count

    async def create(self, *, model, system, tools=None, messages, **params):
        await asyncio.sleep(0)  # stand-in for network I/O
        prefix_text, cacheable = self._extract_prefix(system)
        volatile = sum(len(m["content"]) for m in messages) // 4 + 12  # fake token count
        u = Usage(output_tokens=20)
        if cacheable and prefix_text in self._cache:
            u.cache_read_input_tokens = self._cache[prefix_text]  # HIT — cheap
            u.input_tokens = volatile
        elif cacheable:
            n = max(1, len(prefix_text) // 4)
            self._cache[prefix_text] = n
            u.cache_creation_input_tokens = n  # MISS — pay to write the cache
            u.input_tokens = volatile
        else:
            u.input_tokens = volatile + max(1, len(prefix_text) // 4)  # no caching
        return Message(content=[Block(type="text", text="...answer...")], usage=u, model=model)

    @staticmethod
    def _extract_prefix(system):
        # Anthropic system can be a string OR a list of blocks with cache_control.
        if isinstance(system, list):
            for blk in system:
                if blk.get("cache_control"):
                    return blk["text"], True
            return "".join(b.get("text", "") for b in system), False
        return system or "", False


provider = MockProvider()
print("mock provider ready (simulates a prompt cache)")

## 1 · Prompt caching: a breakpoint after the stable prefix (§11.5)

We mark where the cacheable prefix ends with a `cache_control` breakpoint. The prerequisite is Ch 10's layout: **stable prefix first, volatile content last** — any change invalidates the cache from that point onward. Providers differ in the *trigger* (Anthropic uses explicit breakpoints; OpenAI caches repeated prefixes automatically) but the layout rule is identical.

In [ ]:
LONG_SYSTEM_PROMPT = (
    "You are Acme's support agent. Follow the policy handbook below verbatim.\n"
    + "POLICY: " + ("Refunds within 30 days. Escalate fraud. " * 40)  # a long, STABLE prefix
)

def build_system(prompt):
    # The cache breakpoint: everything up to here is cacheable.
    return [{"type": "text", "text": prompt, "cache_control": {"type": "ephemeral"}}]

async def one_turn(user_text):
    return await provider.create(
        model="claude-sonnet-4-6",
        system=build_system(LONG_SYSTEM_PROMPT),  # stable, cached prefix
        messages=[{"role": "user", "content": user_text}],  # the only part that varies
        max_tokens=256,
    )

print(f"stable prefix is ~{len(LONG_SYSTEM_PROMPT)//4} tokens — worth caching")

### 🔮 Predict: cache hit on the 1st vs the 2nd loop iteration

We run the same agent twice (same system prefix, different user turn). Before running: on iteration **1**, which usage field is non-zero — `cache_creation_input_tokens` or `cache_read_input_tokens`? On iteration **2**? Which one is the cheap one?

In [ ]:
async def two_iterations():
    rows = []
    for i, q in enumerate(["Where is ORD-9912?", "And what's the refund window?"], start=1):
        msg = await one_turn(q)
        rows.append((i, msg.usage.cache_creation_input_tokens, msg.usage.cache_read_input_tokens))
    return rows

rows = asyncio.run(two_iterations())
print(f"{'iter':>4}  {'cache_creation':>14}  {'cache_read':>10}")
for i, create, read in rows:
    print(f"{i:>4}  {create:>14}  {read:>10}")

In [ ]:
# Turn the raw counts into the KPI you actually report: cache hit rate.
total_prefix = sum(c + r for _, c, r in rows)
total_read = sum(r for _, _, r in rows)
hit_rate = total_read / total_prefix if total_prefix else 0.0
print(f"cache hit rate over the run: {hit_rate:.0%}  <- a cost KPI; track it per feature")

# Rough cost model: cached reads bill at ~0.1x of normal input tokens.
READ_DISCOUNT = 0.1
naive_cost = total_prefix              # if nothing were cached, you'd pay full prefill each time
cached_cost = sum(c + r * READ_DISCOUNT for _, c, r in rows)
print(f"prefill cost (relative): naive={naive_cost}  cached={cached_cost:.1f}  "
      f"=> {1 - cached_cost / naive_cost:.0%} cheaper on the prefix")

**What you just saw.** Iteration 1 pays `cache_creation` (you wrote the cache); iteration 2 pays the cheap `cache_read` — the prefill is reused. Over a long agent loop this is routinely the biggest single cost and latency win. Log both fields; **cache hit rate is a cost KPI**.

> ⚠️ **Pitfall — a timestamp in the cacheable prefix.** Let a `datetime.now()`, a per-request id, or a session token slip into the cached prefix and it changes byte-for-byte on every call: the cache never hits, you pay full prefill *plus* the write every time, and the bug is invisible because everything still works. Keep volatile data strictly *after* the breakpoint.

In [ ]:
# Demonstrate the pitfall: a per-request id sneaks into the 'stable' prefix.
async def poisoned_turn(user_text, req_id):
    poisoned = f"[req:{req_id}] " + LONG_SYSTEM_PROMPT  # DON'T: volatile data in the prefix
    return await provider.create(
        model="claude-sonnet-4-6",
        system=build_system(poisoned),
        messages=[{"role": "user", "content": user_text}],
        max_tokens=256,
    )

async def show_poison():
    out = []
    for i in range(2):
        m = await poisoned_turn("hi", req_id=1000 + i)  # different id each call
        out.append(m.usage.cache_read_input_tokens)
    return out

reads = asyncio.run(show_poison())
print("cache_read on each call:", reads, "  <- always 0: the cache NEVER hits. Full prefill, every time.")

## 2 · Batch APIs: half price for work that can wait (§11.5)

Both major providers run asynchronous batch endpoints: submit thousands of requests, collect results within a day (usually much sooner), at roughly half price. Anything offline qualifies — embedding backfills, nightly classification, bulk eval runs (Ch 22), the capstone's corpus-ingestion jobs that Celery schedules (Ch 31). We model a submit/poll/collect lifecycle.

In [ ]:
@dataclass
class BatchJob:
    id: str
    status: str
    results: list = field(default_factory=list)


class MockBatchAPI:
    """Models submit -> poll -> collect at ~half the per-request price."""
    REALTIME_PRICE = 1.0
    BATCH_PRICE = 0.5  # ~half

    def __init__(self):
        self._jobs = {}

    async def submit(self, requests):
        job = BatchJob(id=f"batch_{len(self._jobs):03d}", status="in_progress")
        # Pretend the provider processes them; results are ready on the next poll.
        job.results = [{"custom_id": r["id"], "label": f"classified::{r['text'][:12]}"}
                       for r in requests]
        self._jobs[job.id] = job
        return job.id

    async def poll(self, job_id):
        self._jobs[job_id].status = "completed"  # finishes quickly in the mock
        return self._jobs[job_id].status

    async def collect(self, job_id):
        return self._jobs[job_id].results


# A small list of fake 'offline' requests (the kind you'd never run in real time).
offline_requests = [{"id": f"doc-{i}", "text": f"support ticket number {i} about a refund"}
                    for i in range(8)]
print(f"{len(offline_requests)} offline requests queued for batch")

In [ ]:
async def run_batch(requests):
    api = MockBatchAPI()
    job_id = await api.submit(requests)
    while await api.poll(job_id) != "completed":
        await asyncio.sleep(0)  # real code: sleep + re-poll until done
    results = await api.collect(job_id)
    realtime = len(requests) * api.REALTIME_PRICE
    batched = len(requests) * api.BATCH_PRICE
    return results, realtime, batched

results, realtime, batched = asyncio.run(run_batch(offline_requests))
print("first result:", results[0])
print(f"real-time cost: {realtime}   batch cost: {batched}   "
      f"savings: {1 - batched / realtime:.0%}")
print("Rule: if a workload tolerates hours of latency and you pay real-time prices, you're donating margin.")

## 3 · Provider features worth knowing (§11.6)

The common shape is real, but the divergences are where leverage hides. As of early 2026, a handful of features change what you build yourself vs get for free — and each is a per-provider quirk you want behind *one* adapter:

- **Grounded citations.** Pass documents alongside the question; the model returns answers with first-class citations to the exact passages used — a structural anti-hallucination primitive. Reach for it in RAG/document-Q&A before hand-rolling quote-first prompting.
- **Server-side (hosted) tools.** Some providers run tools *for* you (web search, code execution) so the model can act mid-generation without you securing the executor — convenient, but less control over the sandbox and provider lock-in for that capability.
- **Prompt-cache semantics differ.** Explicit *breakpoints* vs *automatic* prefix caching — but the stable-first layout rule is identical and unforgiving.
- **Reasoning-block round-tripping.** With reasoning models in a tool loop you usually must echo the thinking blocks back on the next turn; silently dropping them degrades multi-step reasoning in ways that are maddening to debug.

All four belong in the adapter you're about to build — not smeared across call sites.

## 4 · 🔧 Build: the single-door `LLMClient`

Now assemble the chapter's centerpiece. One `async call(task, *, messages, tools, **params)` that:
1. resolves model tier + cached system prefix + params from the **prompt registry** (Ch 10, keyed by `task`);
2. wraps the SDK call in a **limiter** — an `asyncio.Semaphore` (concurrency cap) plus a **circuit breaker** that fails fast when the provider is down;
3. records `usage` and `stop_reason` to **metrics**.

No caller knows which provider answered. This is the toy version of [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) — the `usage` fields (`cache_read_input_tokens`, `cache_creation_input_tokens`) match it exactly.

In [ ]:
# --- the pieces the client composes -------------------------------------------

@dataclass
class TaskConfig:
    model: str
    system: str             # the stable, cacheable prefix for this task
    params: dict = field(default_factory=dict)


class PromptRegistry:
    """Stand-in for Ch 10's versioned registry: task name -> model tier + prompt + params."""
    def __init__(self, configs):
        self._configs = configs
    def config(self, task):
        if task not in self._configs:
            raise KeyError(f"unknown task '{task}' — register it in the prompt registry")
        cfg = self._configs[task]
        return cfg
    def system_cached(self, task):
        # Wrap the prefix with a cache breakpoint (see section 1).
        return [{"type": "text", "text": self._configs[task].system,
                 "cache_control": {"type": "ephemeral"}}]


class CircuitOpen(Exception):
    """Raised when the breaker is open: fail fast instead of burning latency."""


class Limiter:
    """Concurrency cap (semaphore) + a simple circuit breaker."""
    def __init__(self, max_concurrency=4, fail_threshold=3):
        self._sem = asyncio.Semaphore(max_concurrency)
        self._fails = 0
        self._threshold = fail_threshold
        self.open = False

    async def __aenter__(self):
        if self.open:
            raise CircuitOpen("provider marked down — failing fast")
        await self._sem.acquire()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        self._sem.release()
        if exc_type is None:
            self._fails = 0  # success resets the breaker
        else:
            self._fails += 1
            if self._fails >= self._threshold:
                self.open = True  # trip the breaker
        return False


class Metrics:
    def __init__(self):
        self.records = []
    def record(self, task, cfg, usage, stop_reason):
        self.records.append({"task": task, "model": cfg.model,
                             "cache_read": usage.cache_read_input_tokens,
                             "cache_creation": usage.cache_creation_input_tokens,
                             "output": usage.output_tokens, "stop_reason": stop_reason})

In [ ]:
# --- the single door ----------------------------------------------------------

class LLMClient:
    """The only door to model APIs in the platform. Construct once, share everywhere.
    Matches the book's §11.7 Build and graduates into blueprints/llm-gateway/."""

    def __init__(self, provider, registry, limiter, metrics):
        self._api = provider        # SDK adapter (Anthropic, OpenAI, ...)
        self._prompts = registry    # versioned templates (Ch 10)
        self._limiter = limiter     # asyncio.Semaphore + circuit breaker
        self._metrics = metrics

    async def call(self, task, *, messages, tools=None, **params):
        cfg = self._prompts.config(task)             # model tier, params, prompt
        async with self._limiter:
            msg = await self._api.create(            # SDK retries + timeout inside
                model=cfg.model,
                system=self._prompts.system_cached(task),  # cached stable prefix
                tools=tools,
                messages=messages,
                **(cfg.params | params),
            )
        self._metrics.record(task, cfg, msg.usage, msg.stop_reason)
        return msg


registry = PromptRegistry({
    "support_reply": TaskConfig(model="claude-sonnet-4-6", system=LONG_SYSTEM_PROMPT,
                                params={"max_tokens": 512}),
    "triage": TaskConfig(model="claude-haiku-4-6", system="Classify the ticket.",
                         params={"max_tokens": 32}),   # cheap tier for the easy step
})
metrics = Metrics()
fresh_provider = MockProvider()  # a cold cache, so this Build section stands on its own
client = LLMClient(fresh_provider, registry, Limiter(), metrics)
print("LLMClient assembled: routing + cached prefix + limiter + metrics behind one call()")

In [ ]:
# Drive a tiny multi-task workload through the ONE door, concurrently.
async def workload():
    tasks = [
        client.call("support_reply", messages=[{"role": "user", "content": "Where is ORD-9912?"}]),
        client.call("support_reply", messages=[{"role": "user", "content": "Refund window?"}]),
        client.call("triage", messages=[{"role": "user", "content": "angry customer, fraud?"}]),
    ]
    await asyncio.gather(*tasks)

asyncio.run(workload())
print(f"{'task':<14}{'model':<20}{'cache_read':>11}{'cache_creation':>15}{'stop':>10}")
for r in metrics.records:
    print(f"{r['task']:<14}{r['model']:<20}{r['cache_read']:>11}{r['cache_creation']:>15}{r['stop_reason']:>10}")

print("\nNo caller knew which provider answered — and every call's usage is now in metrics.")

**What you just saw.** Three calls, two tasks, two model tiers — all through one `call()`. The first `support_reply` wrote the cache (`cache_creation`); the second hit it (`cache_read`); `triage` routed to the cheap model. Every call's `usage` and `stop_reason` landed in `metrics` automatically. *That* is the payoff of the single door: caching, routing, concurrency, and cost visibility are properties of the client, not of forty call sites.

## 🎯 Senior lens

Treat **cost as a quality attribute** (Ch 27) — designed for, measured, and reviewed like latency or reliability. You must be able to answer *"what does one agent run cost, and what drives it?"* from logs, not guesswork. When the answer is too high, you have four standing levers: **caching, batching, routing, and call-count reduction**. And every per-provider quirk — citations, hosted tools, cache-breakpoint placement, reasoning-block round-tripping — belongs in *one* adapter behind a stable interface, where you also record the portability you trade away when you adopt a single-provider feature.

## Recap

- **Prompt caching** reuses the prefill for a stable prefix: pay `cache_creation` once, then cheap `cache_read` on every hit. Layout must be **stable-first, volatile-last**.
- **Cache hit rate is a cost KPI** — log `cache_read` / `cache_creation` on every call.
- A timestamp or per-request id in the cached prefix silently destroys the cache — keep volatile data after the breakpoint.
- **Batch APIs** run offline work at ~half price; anything tolerating hours of latency belongs there.
- Per-provider features (citations, hosted tools, cache semantics, reasoning blocks) live behind **one adapter**.
- The **`LLMClient`** is the single door: routing + cached prefix + limiter (semaphore + circuit breaker) + metrics, so no caller knows which provider answered.

## Exercises

Predict before you run.

1. **Invalidate the cache.** Append a line to `LONG_SYSTEM_PROMPT` *before the breakpoint* between the two iterations of section 1. Predict the `cache_read` on iteration 2, then verify.
2. **Add a model tier.** Register a third task `summarize` on an even cheaper model with `max_tokens=128`. Route a call to it and confirm `metrics` shows the new model. Which of the four cost levers did you just pull?
3. **Trip the breaker.** Make `MockProvider.create` raise on demand, send `fail_threshold` failures through `client.call`, then send one more. Predict the exception on the final call (hint: `CircuitOpen`).
4. **Batch break-even.** At the section-2 prices, how many offline requests must you have before batching saves more than the engineering cost of wiring up submit/poll/collect? State your assumption for that fixed cost.

In [ ]:
# Exercise 1 — your code here

In [ ]:
# Exercise 2 — your code here

In [ ]:
# Exercise 3 — your code here

In [ ]:
# Exercise 4 — your code here

## Next

- ⬅️ **Previous:** [`11-02-resilience-retries-and-idempotency.ipynb`](./11-02-resilience-retries-and-idempotency.ipynb).
- 🔧 **The production version of what you just built:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) — the real wrapper with retries, timeouts, usage logging, prompt caching, model routing, and a circuit breaker. You built the toy; that's the real one.
- 📝 **Template:** the client + `.env` key handling seed [`templates/agent-project-starter/`](../../../templates/agent-project-starter/)'s LLM-access layer.
- 🏗️ **Capstone:** the `LLMClient` is the *only door to model APIs* in the platform; Part IV's agents (`agents/`) are built entirely on top of it — checkpoint `ch11-llm-client`.
- See the book **§11.5–§11.7** for caching, batching, the provider-feature survey, and the full Build.